# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Adapted from your existing exercise. Previous local outputs were cleared so new Colab runs provide their own evidence. 

Source: Original local package: 03_fastai/04_neural_network_foundations.ipynb

In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '04_neural_network_foundations.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


# Lesson 3: how a neural network works
Source task: https://github.com/fastai/course22/blob/master/04-how-does-a-neural-net-really-work.ipynb
This is an independently written solution companion, not a copy of the instructor notebook or a course completion record. Covers fitting a quadratic, gradients, repeated gradient descent, ReLU, and a sum of rectified linear functions.


In [ ]:
import os,json
from pathlib import Path
import numpy as np,pandas as pd,torch
import matplotlib.pyplot as plt
torch.manual_seed(42);torch.set_num_threads(1)
np.random.seed(42)
SMOKE=os.environ.get('COURSE_SMOKE','0')=='1'
results=Path('results');results.mkdir(exist_ok=True)

from torch import nn
x=torch.linspace(-2,2,200)
truth=3*x*x+2*x+1
y=truth+.15*torch.randn_like(x)
coefficients=nn.Parameter(torch.zeros(3))
optimizer=torch.optim.SGD([coefficients],lr=.03)
history=[]
for step in range(250):
    optimizer.zero_grad(set_to_none=True)
    prediction=coefficients[0]*x*x+coefficients[1]*x+coefficients[2]
    loss=(prediction-y).square().mean();loss.backward();optimizer.step()
    history.append(float(loss.detach()))
print('Learned quadratic coefficients:',coefficients.detach())
assert torch.allclose(coefficients.detach(),torch.tensor([3.,2.,1.]),atol=.15)
plt.scatter(x.numpy(),y.numpy(),s=8,label='noisy data')
plt.plot(x.numpy(),prediction.detach().numpy(),color='red',label='fitted quadratic');plt.legend();plt.show()
plt.plot(history);plt.xlabel('Update');plt.ylabel('MSE');plt.show()
def rectified_linear(slope,intercept,x):return torch.relu(slope*x+intercept)
plt.plot(x,rectified_linear(1,0,x),label='ReLU')
plt.plot(x,rectified_linear(-1,-.5,x)+rectified_linear(1,-.5,x),label='two ReLUs');plt.legend();plt.show()
# Gradients accumulate unless zeroed. Demonstrate and then reset them explicitly.
z=torch.tensor(2.,requires_grad=True);(z*z).backward();first=z.grad.clone();(z*z).backward()
assert z.grad==2*first;z.grad.zero_()
pd.DataFrame({'step':range(len(history)),'loss':history}).to_csv(results/'quadratic_training.csv',index=False)

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [ ]:
finish_run(COLAB_CONTEXT)